<a href="https://colab.research.google.com/github/Sarkis55/Datathon2026/blob/main/Datathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# Graduate Assignment: Gender Pay Gap and Prior Pay Analysis
# Google Colab version
# =========================================================

# =========================
# 0. Install / import
# =========================
!pip -q install pandas numpy matplotlib seaborn statsmodels linearmodels scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

sns.set_context("talk")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

# =========================
# 1. Upload or read file
# =========================
# 方法1: 若你已把 graduate-full.csv 放在 Colab 工作目錄
# file_path = "/content/graduate-full.csv"

# 方法2: 若尚未上傳，取消下面註解後手動上傳
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

df_raw = pd.read_csv(file_path)

# 方法2: 若尚未上傳，取消下面註解後手動上傳
# from google.colab import files
# uploaded = files.upload()
# file_path = list(uploaded.keys())[0]

df_raw = pd.read_csv(file_path)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

# =========================
# 2. Basic inspection
# =========================
print("\nColumns:")
print(df_raw.columns.tolist())

print("\nData types:")
print(df_raw.dtypes)

print("\nMissing rate:")
display((df_raw.isna().mean().sort_values(ascending=False) * 100).round(2))

# =========================
# 3. Data cleaning
# =========================
df = df_raw.copy()

# ---- standardize column names just in case
df.columns = [c.strip() for c in df.columns]

# ---- numeric conversion
numeric_cols = [
    "PUBID_1997", "SAMPLE_RACE_1997", "SAMPLE_SEX_1997", "Year",
    "Employed", "TENURE", "HRLY_WAGE", "HRLY_COMP", "HRS_WRK",
    "UID", "Code_1990", "marital_status", "HGC", "Region"
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- date conversion
date_cols = ["DOB", "Interview_Date", "StartDate", "StopDate"]
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

# ---- keep employed observations with valid wages
df = df[df["Employed"] == 1].copy()
df = df[df["HRLY_WAGE"].notna()].copy()
df = df[df["HRLY_WAGE"] > 0].copy()

# ---- optional trimming for extreme hourly wages
# 這裡用 1% 和 99% 分位數 winsorize-like trimming，避免極端值影響
low_w = df["HRLY_WAGE"].quantile(0.01)
high_w = df["HRLY_WAGE"].quantile(0.99)
df = df[(df["HRLY_WAGE"] >= low_w) & (df["HRLY_WAGE"] <= high_w)].copy()

# ---- keep valid hours worked if available
if "HRS_WRK" in df.columns:
    df = df[(df["HRS_WRK"].isna()) | ((df["HRS_WRK"] > 0) & (df["HRS_WRK"] <= 120))].copy()

# ---- keep valid tenure if available
if "TENURE" in df.columns:
    df = df[(df["TENURE"].isna()) | (df["TENURE"] >= 0)].copy()

# =========================
# 4. Feature engineering
# =========================
# Female indicator
# 依資料常見編碼，通常 1=male, 2=female；若你的資料不是，請依實際調整
df["female"] = np.where(df["SAMPLE_SEX_1997"] == 2, 1, 0)

# Log wage
df["ln_wage"] = np.log(df["HRLY_WAGE"])

# Age
if "DOB" in df.columns and "Interview_Date" in df.columns:
    df["age"] = (df["Interview_Date"] - df["DOB"]).dt.days / 365.25
else:
    df["age"] = np.nan

# Squared age
df["age_sq"] = df["age"] ** 2

# Categorical labels
race_map = {
    1: "Hispanic",
    2: "Black",
    3: "Non-Black/Non-Hispanic",
    4: "Mixed/Other"
}
df["race_label"] = df["SAMPLE_RACE_1997"].map(race_map).fillna("Unknown")

sex_map = {1: "Male", 2: "Female"}
df["sex_label"] = df["SAMPLE_SEX_1997"].map(sex_map).fillna("Unknown")

region_map = {
    1: "Northeast",
    2: "North Central",
    3: "South",
    4: "West"
}
df["region_label"] = df["Region"].map(region_map).fillna("Unknown")

# Occupation / industry fallback
for c in ["Occupation_Group2", "Industry_Group", "Occupation", "Industry"]:
    if c in df.columns:
        df[c] = df[c].astype(str).fillna("Unknown")

# Sort for panel-style lag construction
sort_cols = ["PUBID_1997", "Year"]
if "Interview_Date" in df.columns:
    sort_cols.append("Interview_Date")

df = df.sort_values(sort_cols).copy()

# Prior wage: previous observed hourly wage for same individual
df["prior_wage"] = df.groupby("PUBID_1997")["HRLY_WAGE"].shift(1)
df["ln_prior_wage"] = np.log(df["prior_wage"])

# Prior year gap: helps check whether lag is from close period
df["prior_year"] = df.groupby("PUBID_1997")["Year"].shift(1)
df["year_gap"] = df["Year"] - df["prior_year"]

# You can restrict to reasonable gaps if needed
df_lag = df[df["prior_wage"].notna()].copy()
df_lag = df_lag[(df_lag["prior_wage"] > 0)]
df_lag = df_lag[(df_lag["year_gap"].isna()) | (df_lag["year_gap"] <= 3)].copy()

print("Cleaned sample shape:", df.shape)
print("Lag sample shape:", df_lag.shape)

# =========================
# 5. Demographic summary
# =========================
print("\n===== Overall sample summary =====")
display(df[["HRLY_WAGE", "ln_wage", "HRS_WRK", "TENURE", "HGC", "age", "Year"]].describe().T)

print("\n===== Gender distribution =====")
display(df["sex_label"].value_counts(dropna=False).to_frame("count"))

print("\n===== Race distribution =====")
display(df["race_label"].value_counts(dropna=False).to_frame("count"))

print("\n===== Region distribution =====")
display(df["region_label"].value_counts(dropna=False).to_frame("count"))

print("\n===== Distinct individuals =====")
print(df["PUBID_1997"].nunique())

# Year x gender average wage
year_gender = (
    df.groupby(["Year", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"),
           median_wage=("HRLY_WAGE", "median"),
           mean_ln_wage=("ln_wage", "mean"),
           n=("HRLY_WAGE", "size"))
)
display(year_gender.head(20))

# =========================
# 6. Descriptive tests
# =========================
male_wage = df.loc[df["female"] == 0, "HRLY_WAGE"].dropna()
female_wage = df.loc[df["female"] == 1, "HRLY_WAGE"].dropna()

t_stat, p_val = stats.ttest_ind(male_wage, female_wage, equal_var=False, nan_policy="omit")
print("\n===== Mean wage difference test =====")
print("Male mean wage   :", round(male_wage.mean(), 3))
print("Female mean wage :", round(female_wage.mean(), 3))
print("T-statistic      :", round(t_stat, 4))
print("P-value          :", p_val)

# Raw gap
raw_gap_pct = 100 * (female_wage.mean() / male_wage.mean() - 1)
print("Raw female vs male wage gap (%):", round(raw_gap_pct, 2))

# =========================
# 7. Visualizations
# =========================
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="HRLY_WAGE", hue="sex_label", bins=50, kde=True, stat="density", common_norm=False)
plt.title("Hourly Wage Distribution by Gender")
plt.xlabel("Hourly Wage")
plt.ylabel("Density")
plt.show()

plt.figure(figsize=(12, 6))
sns.lineplot(data=year_gender, x="Year", y="mean_wage", hue="sex_label", marker="o")
plt.title("Average Hourly Wage by Gender Over Time")
plt.xlabel("Year")
plt.ylabel("Average Hourly Wage")
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(data=df[df["HRLY_WAGE"] <= df["HRLY_WAGE"].quantile(0.95)], x="sex_label", y="HRLY_WAGE")
plt.title("Hourly Wage by Gender (Trimmed at 95th Percentile for Visualization)")
plt.xlabel("Gender")
plt.ylabel("Hourly Wage")
plt.show()

# Prior wage vs current wage
plot_lag = df_lag[(df_lag["HRLY_WAGE"] <= df_lag["HRLY_WAGE"].quantile(0.99)) &
                  (df_lag["prior_wage"] <= df_lag["prior_wage"].quantile(0.99))].copy()

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=plot_lag.sample(min(8000, len(plot_lag)), random_state=42),
    x="prior_wage", y="HRLY_WAGE", hue="sex_label", alpha=0.35
)
plt.title("Prior Wage vs Current Wage")
plt.xlabel("Prior Wage")
plt.ylabel("Current Wage")
plt.show()

# =========================
# 8. Regression models for gender pay discrimination
# =========================
# Model 1: basic controls
m1 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\n===== Model 1: Gender pay gap with controls =====")
print(m1.summary())

# Model 2: add marital status if available
m2 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\n===== Model 2: Add marital status =====")
print(m2.summary())

# Model 3: add occupation and industry
m3 = smf.ols(
    formula="""
    ln_wage ~ female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df
).fit(cov_type="HC3")

print("\n===== Model 3: Add occupation and industry =====")
print(m3.summary())

# Interpret female coefficient as approximate %
def coef_to_pct(beta):
    return (np.exp(beta) - 1) * 100

print("\n===== Female coefficient interpretation =====")
for name, model in {"M1": m1, "M2": m2, "M3": m3}.items():
    beta = model.params.get("female", np.nan)
    print(f"{name}: female coef = {beta:.4f}, approx % effect = {coef_to_pct(beta):.2f}%")

# =========================
# 9. Prior pay models
# =========================
# Model 4: current wage on prior wage
m4 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage + female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\n===== Model 4: Current wage on prior wage =====")
print(m4.summary())

# Model 5: interaction female * prior wage
m5 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\n===== Model 5: Prior wage interaction with female =====")
print(m5.summary())

# =========================
# 10. Does the prior-current relationship vary over time?
# =========================
# Pre/post 2018 indicator
df_lag["post_2018"] = np.where(df_lag["Year"] >= 2018, 1, 0)

m6 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * post_2018 + female + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\n===== Model 6: Does prior-current wage relationship change after 2018? =====")
print(m6.summary())

# 女性是否在 2018 後有不同變化
m7 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage + female * post_2018 + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\n===== Model 7: Female effect before vs after 2018 =====")
print(m7.summary())

# 三重交互項
m8 = smf.ols(
    formula="""
    ln_wage ~ ln_prior_wage * female * post_2018 + age + age_sq + HGC + TENURE + HRS_WRK
            + C(race_label) + C(region_label) + C(marital_status)
            + C(Occupation_Group2) + C(Industry_Group) + C(Year)
    """,
    data=df_lag
).fit(cov_type="HC3")

print("\n===== Model 8: Triple interaction =====")
print(m8.summary())

# =========================
# 11. Heterogeneity by occupation and industry
# =========================
# Occupation-level female gap
occ_gap = (
    df.groupby(["Occupation_Group2", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"), n=("HRLY_WAGE", "size"))
)

occ_pivot = occ_gap.pivot(index="Occupation_Group2", columns="sex_label", values="mean_wage")
occ_pivot["female_minus_male"] = occ_pivot.get("Female", np.nan) - occ_pivot.get("Male", np.nan)
occ_pivot["female_to_male_ratio"] = occ_pivot.get("Female", np.nan) / occ_pivot.get("Male", np.nan)
occ_pivot = occ_pivot.sort_values("female_to_male_ratio")
display(occ_pivot.head(20))

# Industry-level female gap
ind_gap = (
    df.groupby(["Industry_Group", "sex_label"], as_index=False)
      .agg(mean_wage=("HRLY_WAGE", "mean"), n=("HRLY_WAGE", "size"))
)

ind_pivot = ind_gap.pivot(index="Industry_Group", columns="sex_label", values="mean_wage")
ind_pivot["female_minus_male"] = ind_pivot.get("Female", np.nan) - ind_pivot.get("Male", np.nan)
ind_pivot["female_to_male_ratio"] = ind_pivot.get("Female", np.nan) / ind_pivot.get("Male", np.nan)
ind_pivot = ind_pivot.sort_values("female_to_male_ratio")
display(ind_pivot.head(20))

# Plot occupation ratios
plot_occ = occ_pivot.dropna(subset=["female_to_male_ratio"]).reset_index()
plot_occ = plot_occ.sort_values("female_to_male_ratio").tail(15)

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_occ, y="Occupation_Group2", x="female_to_male_ratio")
plt.axvline(1.0, linestyle="--")
plt.title("Female-to-Male Average Wage Ratio by Occupation Group")
plt.xlabel("Female / Male wage ratio")
plt.ylabel("Occupation Group")
plt.show()

# =========================
# 12. Optional fixed effects model by person
# =========================
# 注意: female 這種不隨時間變動的變數，在個人固定效果模型中會被吸收掉
# 但可用來檢查 prior_wage 關係在同一人內是否成立
# 這邊用 OLS + person dummies，樣本大時可能很慢，可視情況執行

run_fe_model = False

if run_fe_model:
    # 為避免太慢，可先抽樣個體
    sample_ids = df_lag["PUBID_1997"].drop_duplicates().sample(
        min(2000, df_lag["PUBID_1997"].nunique()), random_state=42
    )
    df_fe = df_lag[df_lag["PUBID_1997"].isin(sample_ids)].copy()

    m_fe = smf.ols(
        formula="""
        ln_wage ~ ln_prior_wage + HGC + TENURE + HRS_WRK + C(Year) + C(PUBID_1997)
        """,
        data=df_fe
    ).fit(cov_type="HC3")

    print("\n===== Fixed effects style model =====")
    print(m_fe.summary())

# =========================
# 13. Export regression tables
# =========================
results_table = pd.DataFrame({
    "model": ["M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8"],
    "female_coef": [
        m1.params.get("female", np.nan),
        m2.params.get("female", np.nan),
        m3.params.get("female", np.nan),
        m4.params.get("female", np.nan),
        m5.params.get("female", np.nan),
        m6.params.get("female", np.nan),
        m7.params.get("female", np.nan),
        m8.params.get("female", np.nan),
    ],
    "ln_prior_wage_coef": [
        np.nan,
        np.nan,
        np.nan,
        m4.params.get("ln_prior_wage", np.nan),
        m5.params.get("ln_prior_wage", np.nan),
        m6.params.get("ln_prior_wage", np.nan),
        m7.params.get("ln_prior_wage", np.nan),
        m8.params.get("ln_prior_wage", np.nan),
    ],
    "nobs": [m.nobs for m in [m1, m2, m3, m4, m5, m6, m7, m8]],
    "r2": [m.rsquared for m in [m1, m2, m3, m4, m5, m6, m7, m8]]
})

results_table["female_pct_effect"] = results_table["female_coef"].apply(
    lambda x: (np.exp(x) - 1) * 100 if pd.notna(x) else np.nan
)

display(results_table)
results_table.to_csv("/content/regression_summary.csv", index=False)

# =========================
# 14. Save cleaned files
# =========================
df.to_csv("/content/cleaned_main_sample.csv", index=False)
df_lag.to_csv("/content/cleaned_lag_sample.csv", index=False)

print("\nSaved files:")
print("- /content/cleaned_main_sample.csv")
print("- /content/cleaned_lag_sample.csv")
print("- /content/regression_summary.csv")

# =========================
# 15. Quick interpretation helper
# =========================
print("\n===== Quick interpretation guide =====")
print("""
1. Model 1-3:
   female 係數若顯著為負，代表控制條件後女性薪資仍較低，
   可作為 pay discrimination 的證據之一。

2. Model 4:
   ln_prior_wage 若顯著為正，代表 prior pay 與 current pay 高度正相關。

3. Model 5:
   ln_prior_wage:female 若顯著，代表 prior pay 對女性的影響強度與男性不同。

4. Model 6:
   ln_prior_wage:post_2018 若為負，代表 2018 後 prior-current wage 關係可能變弱。

5. Model 8:
   ln_prior_wage:female:post_2018 若顯著，代表 2018 後 prior pay 對女性的作用出現額外變化。
""")